# Deploy httpbin with `agr`: the shortest complete workflow

This notebook creates a custom Sandbox Tool and one Deployment, accesses httpbin through both a local debugging proxy and the production data-plane domain, and then deletes every resource. Scaling, lifecycle, and session affinity are intentionally left to the adjacent notebooks.

> Every command runs directly. IDs are not extracted automatically: copy each resource ID from the output and set it in the next cell. Notebook outputs are not stored.

## 1. Check the AGR configuration

Confirm that the current credentials and region work. Every later command passes `AGR_REGION` explicitly so the notebook cannot silently use a different local default. Replace `AGR_ROLE_ARN` with a CAM role ARN that allows AGR to pull the target CCR image.

In [ ]:
%env AGR_REGION=ap-shanghai
%env AGR_DOMAIN=tencentags.com
%env AGR_ROLE_ARN=qcs::cam::uin/replace-me:roleName/replace-me
!agr status

## 2. Create the httpbin Sandbox Tool

Replace `your-name` with a unique suffix. The Tool uses the pinned public image and exposes container port `8080` to Deployments. `--wait` waits for the Tool operation to reach a final state.

In [ ]:
%env HTTPBIN_TOOL_NAME=httpbin-simple-your-name
!agr tool create \
  --region "$AGR_REGION" \
  --tool-name "$HTTPBIN_TOOL_NAME" \
  --tool-type custom \
  --persistent \
  --role-arn "$AGR_ROLE_ARN" \
  --network-configuration '{"NetworkMode":"PUBLIC"}' \
  --custom-configuration '{"Image":"ccr.ccs.tencentyun.com/ags.dev/go-httpbin:v2.25.0","ImageRegistryType":"personal","Command":["/bin/go-httpbin"],"Args":["-host","0.0.0.0","-port","8080"],"Env":[{"Name":"EXCLUDE_HEADERS","Value":"X-Access-Token"}],"Ports":[{"Name":"http","Port":8080,"Protocol":"TCP"}],"Resources":{"CPU":"200m","Memory":"500Mi"},"Probe":{"HttpGet":{"Path":"/status/200","Port":8080,"Scheme":"HTTP"},"ReadyTimeoutMs":30000,"ProbeTimeoutMs":1000,"ProbePeriodMs":3000,"SuccessThreshold":1,"FailureThreshold":10}}' \
  --wait

## 3. Create the Deployment

Copy `ToolId` from the preceding output and replace the placeholder below. The Deployment name must be a DNS-1123 name that is unique in the current account. Omitting optional configuration uses the service defaults for scaling and lifecycle.

In [ ]:
%env HTTPBIN_TOOL_ID=sdt-replace-me
%env HTTPBIN_DEPLOYMENT_NAME=httpbin-simple-your-name
!agr deployment create \
  --region "$AGR_REGION" \
  --deployment-name "$HTTPBIN_DEPLOYMENT_NAME" \
  --tool-id "$HTTPBIN_TOOL_ID"

## 4. Inspect the Deployment

Copy `DeploymentId` from the create output. `get` shows the full summary for one Deployment, while `list` shows Deployments in the current region.

In [ ]:
%env HTTPBIN_DEPLOYMENT_ID=dpl-replace-me
!agr deployment get "$HTTPBIN_DEPLOYMENT_ID" --region "$AGR_REGION"
!agr deployment list --region "$AGR_REGION"

## 5. Local debugging: Deployment proxy

`proxy` is a local debugging tool, not the production integration path. The next cell occupies the notebook cell and listens on `127.0.0.1:18080`. Run `curl http://127.0.0.1:18080/get` in another terminal; after verifying the response, interrupt the proxy in the notebook and continue.

In [ ]:
!agr deployment proxy "$HTTPBIN_DEPLOYMENT_ID" 18080:8080 --region "$AGR_REGION"

## 6. Production access: short-lived token and Deployment domain

Production clients should acquire a short-lived Deployment token and call the data-plane domain directly. `AcquireDeploymentToken` is currently invoked through the raw API command. Its `Token` is scoped to the target Deployment and expires at `ExpiresAt`.

The domain rule for a Deployment HTTP port is:

```text
https://{port}-{deployment-id}.{region}.agents.{data-plane-domain}
```

The default data-plane domain is `tencentags.com`, and httpbin listens on `8080`. Never log or commit a real token.

In [ ]:
!agr api call AcquireDeploymentToken \
  --region "$AGR_REGION" \
  --request '{"DeploymentId":"'$HTTPBIN_DEPLOYMENT_ID'"}' \
  --output json

Copy `Data.Response.Response.Token` from the preceding output and set it below. The request carries the token in the `X-Access-Token` header. The Tool configures httpbin not to echo this sensitive header.

In [ ]:
%env HTTPBIN_DEPLOYMENT_TOKEN=dpt-replace-me
!curl --fail-with-body --silent --show-error \
  --header "X-Access-Token: $HTTPBIN_DEPLOYMENT_TOKEN" \
  "https://8080-$HTTPBIN_DEPLOYMENT_ID.$AGR_REGION.agents.$AGR_DOMAIN/get"

## 7. Clean up

Delete the Deployment and wait for its asynchronous deletion before deleting the Tool. Run this section even if an earlier access step failed.

In [ ]:
!agr deployment delete "$HTTPBIN_DEPLOYMENT_ID" --region "$AGR_REGION"
!agr instance list --tool-id "$HTTPBIN_TOOL_ID" --region "$AGR_REGION"

If the list still contains a `RUNNING`, `PAUSED`, or other non-`STOPPED` instance, copy its ID and run the following commands in a new cell. Repeat for every such instance, then delete the Tool.

```text
%env HTTPBIN_INSTANCE_ID=replace-me
!agr instance delete "$HTTPBIN_INSTANCE_ID" --region "$AGR_REGION" --yes --wait
```

In [ ]:
!agr tool delete "$HTTPBIN_TOOL_ID" --region "$AGR_REGION" --yes --wait